## Getting Started with PyleoTUPS: Automated Paleoclimate Data Extraction

**PyleoTUPS** is a Python package designed to streamline paleoclimate data workflows. It automates the extraction and processing of datasets from major repositories like NOAA NCEI.

Key features include:
* **Automated Extraction:** Parses complex text files and templates automatically.
* **Metadata Preservation:** Keeps study-level metadata (authors, location) linked to the data tables.
* **FAIR Compliance:** Supports standards like LiPD and NOAA PaST vocabularies.

In this tutorial, we will cover:
1.  **Searching** for studies using the NOAA API.
2.  **Browsing** search results (Metadata, Sites, Publications).
3.  **Extracting** data into pandas DataFrames.

In [ ]:
import pyleotups as pt

# Initialize the Dataset manager
# This object will act as your session, storing search results and processing data.
dataset = pt.NOAADataset()

### Step 1: Searching for Studies

The primary entry point for finding data is the `search_studies()` method. This connects directly to the NOAA NCEI Paleo Search API.

```Note
Given parameter's are based on the endpoints by NCEI-NOAA and it's description can be found at https://www.ncei.noaa.gov/access/paleo-search/api
```

### 1.1 Search by Identifier
If you know the specific ID of a study, you can fetch it directly. 
*Note: When an ID is provided, all other search parameters (like location or author) are ignored.*

In [ ]:
# Fetch a specific study by its NOAA ID
# This fills the 'dataset' object with the result.
dataset.search_studies(noaa_id=18316)

# View a summary of the result
dataset.get_summary()

### Search by Metadata

For broader discovery, you can search using a combination of parameters. `PyleoTUPS` supports lists for fields like investigators, keywords, and locations.

**Key Parameters:**
* **Geography:** `min_lat`, `max_lat`, `min_lon`, `max_lon`.
* **Investigators:** Provide a list of names. 
    * *Convention:* NOAA expects `"LastName, Initials"` (e.g., `"Wahl, E.R."`). Searching by just `"LastName"` often works but is less precise.
    * *Logic:* By default, multiple items are treated as **OR** (match *any*). Use `investigators_and_or="AND"` to match *all*.
* **Species:** Requires strict **4-letter codes** (e.g., `"PIPO"`. Find Species list [here](https://www.ncei.noaa.gov/access/paleo-search/study/params.json) ) .
* **Time:** `earliest_year`, `latest_year`.
* **Keywords:** Hierarchies are supported (e.g., `"earth science>paleoclimate"`).

##### Efficient Use of `investigator` as a search parameter

In [ ]:
# Clear previous results
dataset = pt.NOAADataset()

# Example 1: "OR" Logic (Default)
# Find studies by EITHER Wahl OR Vose
print("--- Searching for Wahl OR Vose ---")
search_by_investigator_or = dataset.search_studies(
    investigators=["Wahl, E.R.", "Vose, R.S."],
)
print(f"Found {len(search_by_investigator_or)} studies.")

# Example 2: "AND" Logic
# Find studies co-authored by BOTH Wahl AND Vose
# Note: We must specify the `investigators_and_or` parameter
print("\n--- Searching for Wahl AND Vose ---")
search_by_investigator_and = dataset.search_studies(
    investigators=["Wahl, E.R.", "Vose, R.S."],
    investigators_and_or="AND",
)
print(f"Found {len(search_by_investigator_and)} studies.")

In [ ]:
dataset = pt.NOAADataset()

In [ ]:
search_incorrect_investigator_usage = dataset.search_studies(investigators=["E.R., Wahl"])

print(f"Found {len(search_incorrect_investigator_usage)} studies.")

##### Efficient Use of Geographic search parameter

You can search for studies within a specific geographic region using latitude and longitude bounds.

**Parameters:**
* `min_lat`, `max_lat`: Latitude range (Strict Valid: -90 to 90).
* `min_lon`, `max_lon`: Longitude range (Strict Valid: -180 to 180).

**Important Notes:**
1. **Integer Precision:** Decimal values (e.g., `23.5`) will be cast to integers (e.g., `23`) due to NOAA's API requiremnets.
2. **Bounding Box:** The search returns studies that fall *anywhere* within the box defined by these coordinates.

In [ ]:
# Search for studies in a specific region (e.g., Tropical Pacific)
# Latitude: 10S to 10N (-10 to 10)
# Longitude: 120E to 150E (120 to 150)
dataset = pt.NOAADataset()

dataset.search_studies(
    min_lat=-10, max_lat=10,
    min_lon=120, max_lon=150,
    limit = 100
)

# View the location data in the summary
dataset.get_summary()

```Note
PyleoTUPS retrieves and presents the first 100 response from NOAA. You can adjust the ```limit``` parameter to include more results

## Step 2: Browsing Search Results



Once you have performed a search, the `dataset` object holds the results. You can browse these results at three levels of granularity:
1.  **Study Level (`get_summary`)**: High-level metadata (Authors, Titles, Date).
2.  **Site/Table Level (`get_sites`)**: Detailed information about specific sites and the data tables available for them. **This is where you find the IDs needed to extract data.**
3.  **Publication Level (`get_publications`)**: Citation information for the studies.

In [ ]:
# 1. View the high-level summary of the studies found

dataset = pt.NOAADataset()
dataset.search_studies(noaa_id=18316)

summary_df = dataset.get_summary()

display(summary_df)


### Detailed Site & Data Table Information
The `get_sites()` method flattens the hierarchy, providing a row for every data file (or "Data Table") associated with a study.

**Why is this important?**
To extract specific data later, you will need the `dataTableID`. This ID is unique to each file and allows PyleoTUPS to link metadata back to the data.

* **SiteName:** The name of the specific location (e.g., a specific ice core or lake).
* **DataTableID:** The unique identifier for the data file.


In [ ]:
# 2. View detailed site and data table information
sites_df = dataset.get_sites()

# Display columns relevant for data extraction
# Note the 'dataTableID' column - we will use this in Step 3.
display(sites_df)

### Retrieving Citations
The `get_publications()` method returns:
1.  A **BibTeX** object (compatible with reference managers).
2.  A **DataFrame** of publication details (DOIs, years, journals).

**Parameters:**
* `save=True`: Saves a `.bib` file to your disk.
* `verbose=True`: Prints the BibTeX entries to the console.

In [ ]:
# 3. Get citation data
# Returns a tuple: (BibTeX_Database, DataFrame)
bib_db, pubs_df = dataset.get_publications(save=False, verbose=False)

# Display the publication metadata
display(pubs_df)


In [ ]:
# save the BibTeX database to a .bib file. File will be saved in the directory where the notebook/script is run.
dataset.get_publications(save=True)


## Step 3: Extracting Data



This is the core function of PyleoTUPS. Once you have identified the data you want (from the `get_sites()` step above), you can extract it into a pandas DataFrame.

### Method A: By DataTableID (Recommended)
**Why?** When you fetch by ID, PyleoTUPS automatically links the data back to its parent Study and Site. The resulting DataFrame will contain rich metadata in its `.attrs` dictionary.

* **Input:** A string ID or a list of IDs (e.g., `["28694", "28803"]`).
* **Output:** A list of one or more pandas DataFrames.

In [ ]:
# 1. View the high-level summary of the studies found

dataset = pt.NOAADataset()
dataset.search_studies(noaa_id=18316)

dataset.get_sites()


In [ ]:
dfs = dataset.get_data(dataTableIDs=["28694", "28803", "28804"])

In [ ]:
for df in dfs:
    display(df.head())
    print(df.attrs)

NOAA has templated as well as non-templated data tables. The templated tables follow a standard format and include metadata in the table itself, while non-templated tables may have varying formats. 

In [ ]:
speleo8k_ds = pt.NOAADataset()
speleo8k_ds.search_studies(noaa_id=9957)
speleo8k_ds.get_summary()

In [ ]:
speleo8k_ds.get_sites()

In [ ]:
speleo8k_ds_df = speleo8k_ds.get_data(dataTableIDs= ["18803"])

In [ ]:
for df in speleo8k_ds_df:
    display(df.head())
    print(df.attrs)

### Method B: By Direct File URL
If you have a direct link to a NOAA text file (e.g., from an external paper or email), you can fetch it directly without searching first.

**Limitation:** Unless the URL is already known to the `dataset` object (via a previous search), PyleoTUPS **cannot** attach study-level metadata (like Study Name or Site Location) to the DataFrame. You will get the data, but `df.attrs` may be empty or limited to the file header.

In [ ]:
dataset = pt.NOAADataset()
df_data_by_url = dataset.get_data(file_urls="https://www.ncei.noaa.gov/pub/data/paleo/climate_forcing/trace_gases/mcelwain1995co2.txt")

In [ ]:
for df in df_data_by_url:
    display(df.head())
    print(df.attrs) # will have empty attrs

```Note
Only a certain .txt files are supported by pyleotups currently. 

Files which are written as per NOAA Template, or have a NOAA Header can be processed by PyleoTups's parser.
```

In [ ]:
error_ds = pt.NOAADataset()
error_ds.get_data(file_urls = ["https://www.ncei.noaa.gov/pub/data/paleo/reconstructions/climate12k/temperature/version1.0.0/Temp12k_directory_LiPD_files/AdelaideTarn.Jara.2015.lpd"])



In [ ]:
error_ds.get_data(file_urls = ["https://www.ncei.noaa.gov/pub/data/paleo/contributions_by_author/frank1999/frank1999.xls"])

## Step 4: Advanced Usage

### 1. Handling Large Searches (Pagination)
By default, `search_studies` returns up to 100 results. If you need to retrieve a large number of studies, or if you want to process them in batches, you can use the `limit` and `skip` parameters.

* `limit`: The maximum number of studies to return in one request.
* `skip`: The number of studies to skip from the beginning of the result set.

In [ ]:
# Initialize a clean dataset
dataset = pt.NOAADataset()

# Query 1: Get the first 10 results
print("--- Fetching Batch 1 (1-10) ---")
batch1 = dataset.search_studies(min_lat=60, max_lat=70, limit=10)
print(f"Current studies in dataset: {len(dataset.studies)}")
display(batch1)


In [ ]:
# Query 2: Get the next 10 results (skip the first 10)
# Note: This *overwrites* the previous search results in the 'dataset' object as we are reusing the same dataset instance. 
print("\n--- Fetching Batch 2 (11-20) ---")
batch2 = dataset.search_studies(min_lat=60, max_lat=70, limit=10, skip=10)
print(f"Current studies in dataset: {len(dataset.studies)}")
display(batch2)


### 2. Combining Datasets


A powerful feature of PyleoTUPS is the ability to merge `Dataset` objects. This allows you to construct complex collections of data that might be difficult to express in a single search query.

For example, you can perform two separate searches (e.g., one for "Pollen" and one for "Ice Cores") and combine them into a single dataset for analysis.

* `+` operator: Creates a new Dataset containing the union of two others.
* `+=` operator: Adds results from one Dataset into another (in-place).

In [ ]:
# Create two separate datasets
ds_pollen = pt.NOAADataset()
ds_ice = pt.NOAADataset()

# Search 1: Pollen studies in a specific region
print("Searching for Pollen...")
search_pollen = ds_pollen.search_studies(keywords=["pollen"], min_lat=40, max_lat=50, limit=5)
display(search_pollen[['StudyName', 'StudyID']])

# Search 2: Ice Core studies in the same region
print("Searching for Ice Cores...")
search_ice = ds_ice.search_studies(keywords=["ice core"], min_lat=40, max_lat=50, limit=5)
display(search_ice[['StudyName', 'StudyID']])

# Combine them into a master dataset
ds_combined = ds_pollen + ds_ice

print(f"\nPollen studies: {len(ds_pollen.studies)}")
print(f"Ice Core studies: {len(ds_ice.studies)}")
print(f"Combined total: {len(ds_combined.studies)}")

# Verify the combination by looking at the summary
ds_combined.get_summary()[['StudyName', 'StudyID']].head(10)

## STAY TUNED FOR MORE:

Check the upcoming releases for:
- Data Extraction from Excel Files
- Dataset access from PANAGAEA

### Excel Parser:

In [ ]:
from pyleotups.utils.Parser.ExcelParser import ExcelParser

excel_url = "https://www.ncei.noaa.gov/pub/data/paleo/contributions_by_author/frank1999/frank1999.xls"
excel_parser = ExcelParser(excel_url)
excel_data_blocks = excel_parser.parse()

for block in excel_data_blocks:
    if hasattr(block, 'df') and block.df is not None:
        display(block.df.head())